<a href="https://colab.research.google.com/github/sandeepbansal0/Python-Projects/blob/main/Introduction_to_Difference_in_Differences.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Introduction to Difference-in-Differences (DiD) Using Python

## What is Difference-in-Differences?

**Difference-in-Differences (DiD)** is a statistical technique used to estimate the causal impact of a treatment, policy, or business intervention.

It works by comparing changes over time between two groups:

- **Treatment Group:** Receives the intervention.
- **Control Group:** Does not receive the intervention.

DiD is commonly used in:

- Data Science
- Economics
- Marketing Analytics
- Business Experimentation

---

## 1. Business Problem Statement

### Scenario

A telecommunications company wants to determine whether a new marketing campaign increased monthly sales.

### The Two Groups

**Treatment Group**
- Stores that received the marketing campaign.

**Control Group**
- Stores that did not receive the marketing campaign.

### Objective

Measure monthly sales **before and after** the campaign for both groups.

Then determine whether the treatment group experienced a greater increase in sales compared with the control group.

### Research Question

**Did the marketing campaign increase monthly sales beyond what would have happened without the campaign?**

---

## 2. Why Use Difference-in-Differences?

Simply comparing sales before and after a campaign can be misleading.

Sales may increase because of:

- Seasonal demand
- Economic conditions
- Market growth
- Changes in customer behavior

DiD helps account for changes affecting both groups by comparing their sales trends.

**Goal:** Estimate the additional sales increase attributable to the marketing campaign, assuming the DiD assumptions hold.


In [1]:

import pandas as pd

df = pd.DataFrame({
    "group": ["Treatment", "Treatment", "Control", "Control"],
    "period": ["Before", "After", "Before", "After"],
    "sales": [100, 150, 80, 100]
})

print(df)


       group  period  sales
0  Treatment  Before    100
1  Treatment   After    150
2    Control  Before     80
3    Control   After    100


In [2]:

# Reshape the dataset
pivot = df.pivot(
    index="group",
    columns="period",
    values="sales"
)

# Calculate changes
pivot["change"] = pivot["After"] - pivot["Before"]

print(pivot)

# Calculate Difference-in-Differences
did = (
    pivot.loc["Treatment", "change"]
    - pivot.loc["Control", "change"]
)

print("DiD Estimate:", did)


period     After  Before  change
group                           
Control      100      80      20
Treatment    150     100      50
DiD Estimate: 30



The treatment group's sales increased by **$50**, but the control group's sales also increased by **$20**.

We therefore cannot attribute the entire $50 increase to the campaign.

## 2. Difference-in-Differences Calculation

### Formula

$$
DiD = (T_{after} - T_{before}) - (C_{after} - C_{before})
$$

### Substituting Our Values

$$
DiD = (150 - 100) - (100 - 80)
$$

$$
DiD = 50 - 20
$$

$$
\boxed{DiD = 30}
$$

### Interpretation

The marketing campaign is estimated to have increased monthly sales by **$30 per store**, beyond the increase observed in the control group, assuming the DiD assumptions hold.


In [3]:

import pandas as pd
import statsmodels.formula.api as smf

df["treatment"] = (df["group"] == "Treatment").astype(int)
df["post"] = (df["period"] == "After").astype(int)

model = smf.ols(
    "sales ~ treatment + post + treatment:post",
    data=df
).fit()

print(model.params)


Intercept         80.0
treatment         20.0
post              20.0
treatment:post    30.0
dtype: float64



## 5. Estimate Difference-in-Differences Using Regression

Difference-in-Differences (DiD) can also be estimated using **linear regression**.

### Regression Equation

$$
Sales = \beta_0 + \beta_1 Treatment + \beta_2 Post + \beta_3(Treatment \times Post) + \epsilon
$$

### Understanding the Variables

- **Sales:** Outcome variable (monthly sales).
- **β₀ (Intercept):** Average sales for the control group before the campaign.
- **β₁ (Treatment):** Initial difference in sales between treatment and control groups.
- **β₂ (Post):** Change in sales for the control group after the campaign.
- **β₃ (Treatment × Post):** Difference-in-Differences treatment effect.
- **ε (Error Term):** Unexplained variation in sales.

### Key Concept

The interaction coefficient **β₃** represents our Difference-in-Differences estimate.

$$
\boxed{\beta_3 = DiD}
$$

In our example:

$$
\boxed{\beta_3 = 30}
$$

### Interpretation

The marketing campaign is estimated to have increased average monthly sales by **$30 per store**, relative to the change in the control group, assuming the DiD assumptions hold.



## 5. Regression Results and Interpretation

### Expected Regression Coefficients

**1. Intercept (β₀)**

- Coefficient: **80**
- Interpretation: Average sales for the control group before the marketing campaign.

**2. Treatment (β₁)**

- Coefficient: **20**
- Interpretation: Initial difference in sales between treatment and control groups before the campaign.

**3. Post (β₂)**

- Coefficient: **20**
- Interpretation: Change in sales for the control group after the campaign.

**4. Treatment × Post (β₃)**

- Coefficient: **30**
- Interpretation: Estimated additional sales generated by the marketing campaign.

### Key Takeaway

The interaction coefficient (β₃) represents the Difference-in-Differences estimate.

$$
\boxed{DiD = \beta_3 = 30}
$$

**Important Note:**

This four-row example is a saturated regression model.

- It demonstrates how regression coefficients are calculated.
- It cannot provide meaningful standard errors.
- It cannot support meaningful statistical significance tests.
- Real-world analysis requires multiple observations per group and period for reliable statistical inference.

---

## 6. Important Assumption: Parallel Trends

### What Are Parallel Trends?

The most important assumption in Difference-in-Differences is **parallel trends**.

This means that, without the marketing campaign, the treatment and control groups would have experienced similar changes in sales over time.

### Example

Suppose treatment stores were already experiencing faster sales growth than control stores before the campaign.

If this existing growth pattern continued, Difference-in-Differences could incorrectly attribute some of that growth to the marketing campaign.

### How to Evaluate Parallel Trends

**1. Examine Historical Data**

- Collect several periods of data before the intervention.
- Compare sales trends between treatment and control groups.

**2. Visualize Pre-Treatment Trends**

- Plot treatment and control sales over time.
- Look for similar patterns of growth or decline.

**3. Investigate Other Factors**

- Seasonal demand
- Market conditions
- Competitor activity
- Other marketing campaigns
- Changes in customer behavior

**4. Use Appropriate Statistical Methods**

- Estimate DiD using regression.
- Consider statistical uncertainty.
- Use appropriate standard errors, often clustered at the level where treatment was assigned.

### Key Takeaway

**Parallel trends means both groups would have followed similar changes over time if the intervention had not occurred.**

Historical data can help evaluate whether this assumption is plausible, but cannot prove it.

---

## 7. When Should You Use Difference-in-Differences?

DiD is useful when you want to estimate the causal effect of a business intervention and have comparable treatment and control groups observed before and after the intervention.

### Example 1: Marketing Campaign Effectiveness

**Business Question:**

Did a marketing campaign increase sales?

**Treatment Group:**
- Stores receiving the marketing campaign.

**Control Group:**
- Stores not receiving the marketing campaign.

**Outcome Variable:**
- Monthly sales.

---

### Example 2: Digital App Feature

**Business Question:**

Did a new app feature improve customer conversion?

**Treatment Group:**
- Customers exposed to the new feature.

**Control Group:**
- Customers not exposed to the new feature.

**Outcome Variable:**
- Conversion rate.

---

### Example 3: Employee Training

**Business Question:**

Did a new training program improve employee productivity?

**Treatment Group:**
- Employees who received training.

**Control Group:**
- Employees who did not receive training.

**Outcome Variable:**
- Employee productivity.

---

### Example 4: Customer Retention Program

**Business Question:**

Did a new retention program reduce customer churn?

**Treatment Group:**
- Customers receiving the retention program.

**Control Group:**
- Comparable customers not receiving the program.

**Outcome Variable:**
- Customer churn rate.

---

## 8. Final Takeaways

1. **Difference-in-Differences** estimates the effect of an intervention by comparing changes between treatment and control groups.

2. **Regression** provides another way to calculate the DiD estimate.

3. **The interaction coefficient (β₃)** represents the estimated treatment effect.

4. **Parallel trends** is a critical assumption for causal interpretation.

5. **Multiple observations** are needed for meaningful statistical inference.

6. **Business applications** include marketing, customer retention, conversion optimization, and employee productivity.

### Final Formula

$$
\boxed{DiD = \Delta Treatment - \Delta Control}
$$

**Bottom Line:**

Difference-in-Differences helps estimate whether an intervention produced an additional change in an outcome beyond what would have been expected without the intervention, provided its assumptions hold.


A telecommunications company introduces a new digital sales tool in selected markets.

Group	Before	After
Treatment
200	280
Control
150	190

What is the Difference-in-Differences estimate?

Treatment change = 280 − 200 = 80.

Control change = 190 − 150 = 40.

DiD = 80 − 40 = 40.